# 03f · 복구된 03F private 평가 코드

이 파일은 DCC 2026 작업 노트북의 **공개용 보존본**입니다. 아래 원래 셀과 텍스트 출력은 저장된 실험 기록이며, 이번 공개 작업에서 모델 학습·추론·평가를 다시 실행하지 않았습니다.

- **목적:** 03E 예측 hash를 검증한 뒤 private key map과 정답을 결합해 RMSE·오예측 분석을 수행하도록 작성된 과거 코드입니다.
- **입력:** 03E prediction seal, private key map·test truth와 필요한 익명 shard
- **출력:** 실행 시 private RMSE 표·잔차·오류 표본·평가 seal
- **실제 저장된 실행 상태:** outputs/improvement_cycle9/recovered/03F_drive.ipynb에서 복구한 코드입니다. 저장 출력은 0개이며 실행 여부와 결과는 검증되지 않았습니다. 재현 경로를 이해하는 선택적 과거 자료이며 03E→03F의 성공 실행 증거로 사용하지 않습니다. 누락된 01R을 대체하지 않습니다.

**실행 전:** 공개 저장소에는 원천 이미지·라벨 ZIP, private test 연결표·정답, 모델 가중치, 원래 GCS 산출물이 없습니다. `YOUR_GCP_PROJECT_ID`, `YOUR_GCS_BUCKET`과 필요한 데이터·권한을 직접 설정해야 하며, 원래 봉인 파일의 generation·SHA-256·config 검증도 충족해야 합니다. placeholder만 바꾼다고 과거 산출물이 재생성되거나 모든 gate가 통과하는 것은 아닙니다.

**공개본 변경:** 개인 프로젝트·버킷·계정·로컬 데이터 경로를 placeholder로 바꾸고 Colab/Drive/작성자/widget metadata를 제거했습니다. 코드 셀의 순서와 계산·무결성 검증 로직은 보존했습니다. 모든 내장 image/HTML/JavaScript/기타 binary 출력과 markdown 첨부 이미지를 제거하고, 숫자·표·오류를 포함한 텍스트 출력은 식별자를 가린 뒤 보존했습니다. 사진이 없는 것은 해당 셀을 실행하지 않았다는 뜻이 아닙니다. 원본 code cell의 `execution_count`는 모두 `null`이므로 이 값만으로 실행 여부를 판단하지 않습니다.

[GitHub 코드](https://github.com/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/historical/03f_private_evaluation_recovered.ipynb) · [Colab에서 열기](https://colab.research.google.com/github/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/historical/03f_private_evaluation_recovered.ipynb) · [실행 관계와 상태](../README.md) · [원본 이름·SHA-256 대응표](../../docs/notebook-source-map.csv)

---

아래는 비공개 식별자를 치환한 원래 노트북입니다. 원래 안내의 예정 단계·성공 기준은 위의 실제 저장 상태와 구분해 읽으십시오.


# DCC 2026 Mission 3 — 03F. Private test RMSE와 오예측 분석

이 노트북은 03E의 prediction seal과 SHA-256을 먼저 검증한 뒤 private key map과
`test_truth_v3`를 처음으로 결합합니다.

- 전체 test RMSE와 A/B/C/E/F/G/H/S RMSE를 meter로 계산
- 유형별 n·MAE·bias·95% 절대오차를 함께 기록
- 미리 고정한 규칙으로 오예측 표본을 선택
- obstruction, duplicate target conflict, fallback은 **사후 분석 정보**로만 사용


## 런타임

표준 CPU 런타임을 사용합니다. 모델 학습·추론은 수행하지 않습니다. 오예측 montage를 만들기
위해 익명 test shard 일부를 다시 내려받을 수 있습니다.


In [ ]:
%pip -q install "google-cloud-storage>=2.19,<4" "pyarrow>=18,<24" "pandas>=2.2,<3" "matplotlib>=3.8,<4" "pillow>=10,<13"


In [ ]:
import hashlib
import io
import json
import math
import tarfile
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from google.colab import auth
from PIL import Image

PROJECT_ID = "YOUR_GCP_PROJECT_ID"
BUCKET_NAME = "YOUR_GCS_BUCKET"
PREDICTION_RUN_ID = "m3-test-prediction-router-full-v2"
SELECTION_ID = "m3-selection-full-router-v2"
EVALUATION_ID = "m3-private-evaluation-router-full-v2"
SPLIT_VERSION = "v3"
RAW_MANIFEST_VERSION = "v2"
EXPECTED_TEST_N = 30_809
EXPECTED_TEST_TYPES = {
    "A": 10_522, "B": 3_727, "C": 9_194, "E": 2_351,
    "F": 878, "G": 1_098, "H": 1_031, "S": 2_008,
}
TYPE_ORDER = list("ABCEFGHS")
BUILD_ERROR_MONTAGE = True

LOCAL_ROOT = Path("/content/dcc_mission3_03f")
LOCAL_INPUT = LOCAL_ROOT / "input"
LOCAL_OUTPUT = LOCAL_ROOT / "output"
LOCAL_FIGURES = LOCAL_OUTPUT / "figures"
LOCAL_META = LOCAL_ROOT / "meta"
for path in [LOCAL_INPUT, LOCAL_OUTPUT, LOCAL_FIGURES, LOCAL_META]:
    path.mkdir(parents=True, exist_ok=True)

PREDICTION_PREFIX = f"dcc/inference/mission3/predictions/{PREDICTION_RUN_ID}"
EVAL_PREFIX = f"dcc/eval_private/mission3/evaluations/{EVALUATION_ID}"


In [ ]:
from google.api_core.exceptions import NotFound, PreconditionFailed
from google.cloud import storage

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def canonical_sha256(value):
    raw = json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def canonical_target_sha256(frame, target_column):
    """원본 float64와 DataLoader float32가 같은 수위를 같은 byte 계약으로 해시합니다."""
    required = {"stem", "type", target_column}
    missing = required - set(frame.columns)
    if missing:
        raise AssertionError(f"target hash 필수 열 누락: {sorted(missing)}")
    ordered = frame.sort_values("stem", kind="mergesort")
    records = [
        {
            "stem": str(getattr(row, "stem")),
            "type": str(getattr(row, "type")),
            "target_m_float32": float(np.float32(float(getattr(row, target_column)))),
        }
        for row in ordered.itertuples(index=False)
    ]
    return canonical_sha256(records)

def parse_gs_uri(uri):
    if not str(uri).startswith("gs://"):
        raise ValueError(f"GCS URI가 아닙니다: {uri}")
    bucket_name, object_name = str(uri)[5:].split("/", 1)
    return bucket_name, object_name

def download_object(bucket, object_name, local_path, generation=None):
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    blob = bucket.blob(object_name, generation=generation)
    kwargs = {"checksum": "auto"}
    if generation is not None:
        kwargs["if_generation_match"] = int(generation)
    blob.download_to_filename(str(local_path), **kwargs)
    return local_path

def upload_create_only(bucket, local_path, object_name, artifact_type):
    local_path = Path(local_path)
    file_sha = sha256_file(local_path)
    blob = bucket.blob(object_name)
    blob.metadata = {
        "sha256": file_sha,
        "artifact_type": artifact_type,
    }
    try:
        blob.upload_from_filename(
            str(local_path), if_generation_match=0, checksum="auto"
        )
    except PreconditionFailed:
        blob.reload()
        remote_sha = (blob.metadata or {}).get("sha256")
        remote_artifact_type = (blob.metadata or {}).get("artifact_type")
        if (
            remote_sha != file_sha
            or remote_artifact_type != artifact_type
            or int(blob.size or -1) != local_path.stat().st_size
        ):
            raise RuntimeError(
                f"create-only 충돌: gs://{bucket.name}/{object_name}"
            )
    blob.reload()
    if (
        (blob.metadata or {}).get("sha256") != file_sha
        or (blob.metadata or {}).get("artifact_type") != artifact_type
    ):
        raise RuntimeError(f"업로드 SHA metadata 검증 실패: {object_name}")
    return {
        "uri": f"gs://{bucket.name}/{object_name}",
        "generation": int(blob.generation),
        "bytes": int(blob.size),
        "sha256": file_sha,
        "artifact_type": artifact_type,
    }

def upload_json_create_only(bucket, payload, object_name, artifact_type):
    local_path = LOCAL_META / Path(object_name).name
    local_path.parent.mkdir(parents=True, exist_ok=True)
    local_path.write_text(
        json.dumps(
            payload, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False
        ) + "\n",
        encoding="utf-8",
    )
    return upload_create_only(bucket, local_path, object_name, artifact_type)

def download_json(bucket, object_name, generation=None):
    blob = bucket.blob(object_name, generation=generation)
    kwargs = {}
    if generation is not None:
        kwargs["if_generation_match"] = int(generation)
    return json.loads(blob.download_as_bytes(**kwargs).decode("utf-8"))

def remote_descriptor(bucket, object_name, artifact_type=None):
    blob = bucket.blob(object_name)
    blob.reload()
    metadata = blob.metadata or {}
    file_sha = metadata.get("sha256")
    if not file_sha:
        raise RuntimeError(f"remote object에 sha256 metadata가 없습니다: {object_name}")
    remote_artifact_type = metadata.get("artifact_type")
    if artifact_type is not None and remote_artifact_type != artifact_type:
        raise RuntimeError(
            f"remote artifact_type 불일치: {object_name} "
            f"({remote_artifact_type!r} != {artifact_type!r})"
        )
    return {
        "uri": f"gs://{bucket.name}/{blob.name}",
        "generation": int(blob.generation),
        "bytes": int(blob.size),
        "sha256": file_sha,
        "artifact_type": remote_artifact_type or "unknown",
    }


## 1. prediction seal과 prediction byte를 먼저 검증

이 셀이 통과하기 전에는 private truth를 다운로드하지 않습니다.


In [ ]:
auth.authenticate_user()
STORAGE_CLIENT = storage.Client(project=PROJECT_ID)
BUCKET = STORAGE_CLIENT.bucket(BUCKET_NAME)
BUCKET.reload()

prediction_seal_object = f"{PREDICTION_PREFIX}/prediction_seal.json"
prediction_seal_path = LOCAL_INPUT / "prediction_seal.json"
download_object(BUCKET, prediction_seal_object, prediction_seal_path)
prediction_seal_sha256 = sha256_file(prediction_seal_path)
prediction_seal = json.loads(prediction_seal_path.read_text(encoding="utf-8"))
if prediction_seal["status"] != "TEST_PREDICTIONS_FROZEN":
    raise AssertionError("test prediction이 동결되지 않았습니다")
prediction_assertions = prediction_seal.get("assertions", {})
if not prediction_assertions or not all(prediction_assertions.values()):
    raise AssertionError("prediction seal assertion 실패")
if prediction_seal.get("truth_accessed") is not False:
    raise AssertionError("inference runtime에서 truth 접근 흔적이 있습니다")
if prediction_seal.get("selection_id") != SELECTION_ID:
    raise AssertionError("full selection ID 불일치")
if prediction_seal.get("prediction_run_id") != PREDICTION_RUN_ID:
    raise AssertionError("prediction run ID 불일치")
if prediction_seal.get("training_scope") != "full":
    raise AssertionError("full 학습 prediction이 아닙니다")
if prediction_seal.get("split_version") != SPLIT_VERSION:
    raise AssertionError("prediction split version 불일치")
if prediction_seal.get("schema_version") != "dcc-m3-prediction-seal-v2":
    raise AssertionError("prediction seal schema 오류")
if prediction_seal.get("winner_kind") != "type_router":
    raise AssertionError("동결된 type-router prediction이 아닙니다")
routing_contract = prediction_seal.get("routing_contract")
if not routing_contract:
    raise AssertionError("prediction routing contract 누락")
if canonical_sha256(routing_contract) != prediction_seal.get("routing_sha256"):
    raise AssertionError("prediction routing SHA 불일치")
route_by_type = dict(routing_contract.get("route_by_type", {}))
if set(route_by_type) != set(TYPE_ORDER):
    raise AssertionError("prediction route type 집합 오류")
if not set(route_by_type.values()).issubset({"full_frame", "yolo_roi"}):
    raise AssertionError("prediction route component 오류")
expected_route_counts = {
    component_name: int(sum(
        EXPECTED_TEST_TYPES[type_name]
        for type_name, routed in route_by_type.items()
        if routed == component_name
    ))
    for component_name in ["full_frame", "yolo_roi"]
}
if prediction_seal.get("route_counts") != expected_route_counts:
    raise AssertionError("prediction route count 계약 불일치")

def verified_download_descriptor(descriptor, local_name):
    bucket_name, object_name = parse_gs_uri(descriptor["uri"])
    if bucket_name != BUCKET_NAME:
        raise AssertionError("artifact bucket 불일치")
    local = LOCAL_INPUT / local_name
    download_object(BUCKET, object_name, local, generation=descriptor["generation"])
    expected_bytes = int(descriptor.get("bytes", descriptor.get("size", -1)))
    if local.stat().st_size != expected_bytes:
        raise AssertionError("artifact size 불일치")
    if sha256_file(local) != descriptor["sha256"]:
        raise AssertionError("artifact SHA 불일치")
    return local

prediction_path = verified_download_descriptor(
    prediction_seal["prediction"], "test_predictions.csv"
)
diagnostics_path = verified_download_descriptor(
    prediction_seal["diagnostics"], "test_diagnostics.parquet"
)
test_use_lease_path = verified_download_descriptor(
    prediction_seal["test_use_lease"], "test_use_lease.json"
)
inference_image_seal_path = verified_download_descriptor(
    prediction_seal["inference_image_seal"], "test_image_seal.json"
)
if sha256_file(inference_image_seal_path) != prediction_seal["inference_image_seal_sha256"]:
    raise AssertionError("prediction inference image seal SHA 불일치")
inference_image_seal = json.loads(
    inference_image_seal_path.read_text(encoding="utf-8")
)
inference_descriptor = prediction_seal["inference_image_seal"]
_, inference_object_name = parse_gs_uri(inference_descriptor["uri"])
expected_inference_object = (
    f"dcc/inference/mission3/{prediction_seal['dataset_version']}/test_image_seal.json"
)
if (
    inference_descriptor.get("artifact_type") != "m3_inference_image_seal"
    or inference_object_name != expected_inference_object
):
    raise AssertionError("inference image seal prefix/artifact type 불일치")
if inference_image_seal.get("status") != "INFERENCE_IMAGES_COMPLETE":
    raise AssertionError("inference image seal 상태 오류")
if inference_image_seal.get("dataset_version") != prediction_seal["dataset_version"]:
    raise AssertionError("prediction과 inference image dataset version 불일치")
if inference_image_seal.get("split_seal_sha256") != prediction_seal["split_seal_sha256"]:
    raise AssertionError("prediction과 inference image split seal 불일치")
if inference_image_seal.get("test_sample_id_sha256") != prediction_seal["test_sample_id_sha256"]:
    raise AssertionError("prediction과 inference image sample ID SHA 불일치")
inference_assertions = inference_image_seal.get("assertions", {})
if not inference_assertions or not all(inference_assertions.values()):
    raise AssertionError("inference image seal assertion 실패")
test_use_lease = json.loads(test_use_lease_path.read_text(encoding="utf-8"))
lease_descriptor = prediction_seal["test_use_lease"]
_, lease_object_name = parse_gs_uri(lease_descriptor["uri"])
expected_lease_object = (
    "dcc/inference/mission3/test_use_leases/"
    f"split-{prediction_seal['split_seal_sha256']}.json"
)
if (
    lease_descriptor.get("artifact_type") != "m3_test_use_lease"
    or lease_object_name != expected_lease_object
):
    raise AssertionError("test-use lease prefix/artifact type 불일치")
expected_lease_fields = {
    "status": "TEST_USE_LOCKED_TO_DEPLOYMENT",
    "dataset_version": prediction_seal["dataset_version"],
    "split_version": prediction_seal["split_version"],
    "split_seal_sha256": prediction_seal["split_seal_sha256"],
    "prediction_run_id": prediction_seal["prediction_run_id"],
    "selection_id": prediction_seal["selection_id"],
    "selection_seal_sha256": prediction_seal["selection_seal_sha256"],
    "winner_model_id": prediction_seal["winner_model_id"],
    "winner_kind": prediction_seal["winner_kind"],
    "checkpoint_bundle_sha256": prediction_seal["checkpoint_bundle_sha256"],
    "routing_sha256": prediction_seal["routing_sha256"],
    "inference_contract_sha256": prediction_seal["inference_contract_sha256"],
}
lease_mismatches = {
    key: (test_use_lease.get(key), expected)
    for key, expected in expected_lease_fields.items()
    if test_use_lease.get(key) != expected
}
if lease_mismatches:
    raise AssertionError(f"test-use lease field 불일치: {lease_mismatches}")

inference_contract = prediction_seal.get("inference_contract", {})
if canonical_sha256(inference_contract) != prediction_seal["inference_contract_sha256"]:
    raise AssertionError("prediction inference contract SHA 불일치")
expected_contract_fields = {
    "schema_version": "dcc-m3-test-inference-contract-v1",
    "prediction_run_id": prediction_seal["prediction_run_id"],
    "selection_id": prediction_seal["selection_id"],
    "selection_seal_sha256": prediction_seal["selection_seal_sha256"],
    "inference_image_seal": prediction_seal["inference_image_seal"],
    "inference_image_seal_sha256": prediction_seal["inference_image_seal_sha256"],
    "dataset_seal_sha256": prediction_seal["dataset_seal"]["sha256"],
    "split_seal_sha256": prediction_seal["split_seal_sha256"],
    "winner_model_id": prediction_seal["winner_model_id"],
    "winner_kind": prediction_seal["winner_kind"],
    "checkpoint_bundle_sha256": prediction_seal["checkpoint_bundle_sha256"],
    "routing_sha256": prediction_seal["routing_sha256"],
}
contract_mismatches = {
    key: (inference_contract.get(key), expected)
    for key, expected in expected_contract_fields.items()
    if inference_contract.get(key) != expected
}
if contract_mismatches:
    raise AssertionError(
        f"prediction inference contract field 불일치: {contract_mismatches}"
    )
predictions = pd.read_csv(prediction_path).sort_values(
    "sample_id", kind="mergesort"
).reset_index(drop=True)
diagnostics = pd.read_parquet(diagnostics_path).sort_values(
    "sample_id", kind="mergesort"
).reset_index(drop=True)
assert list(predictions.columns) == ["sample_id", "type", "pred_m"]
assert len(predictions) == len(diagnostics) == EXPECTED_TEST_N
assert predictions["sample_id"].is_unique
assert diagnostics["sample_id"].is_unique
assert predictions["sample_id"].tolist() == diagnostics["sample_id"].tolist()
assert (
    canonical_sha256(sorted(predictions["sample_id"].astype(str).tolist()))
    == prediction_seal["test_sample_id_sha256"]
)
assert "route_component" in diagnostics.columns
assert np.isfinite(predictions["pred_m"].to_numpy(float)).all()
assert predictions["type"].value_counts().to_dict() == EXPECTED_TEST_TYPES
assert predictions["type"].tolist() == predictions["sample_id"].str[0].tolist()
expected_routes = predictions["type"].map(route_by_type)
assert diagnostics["route_component"].tolist() == expected_routes.tolist()
assert diagnostics["route_component"].value_counts().to_dict() == expected_route_counts
assert not diagnostics.loc[
    diagnostics["route_component"].eq("full_frame"), "fallback_used"
].astype(bool).any()
print("PREDICTION SEAL VERIFIED — private truth download is now allowed")


## 2. private key map과 test truth 결합

예측의 익명 `sample_id`를 원래 stem으로 복원한 다음 `test_truth_v3`와 one-to-one outer join을
수행합니다. stem 첫 문자와 inference type도 이 단계에서 교차 확인합니다.


In [ ]:
DATASET_VERSION = prediction_seal["dataset_version"]

# prediction seal이 가리킨 exact dataset seal generation을 먼저 검증합니다. 그 seal 안의
# private key-map descriptor와 upstream split seal descriptor만 후속 입력으로 허용합니다.
dataset_seal_path = verified_download_descriptor(
    prediction_seal["dataset_seal"], "dataset_seal.json"
)
dataset_seal = json.loads(dataset_seal_path.read_text(encoding="utf-8"))
if dataset_seal.get("status") != "DATASET_COMPLETE":
    raise AssertionError("dataset seal 상태 오류")
if dataset_seal.get("dataset_version") != DATASET_VERSION:
    raise AssertionError("prediction과 dataset seal version 불일치")
dataset_assertions = dataset_seal.get("assertions", {})
if not dataset_assertions or not all(dataset_assertions.values()):
    raise AssertionError("dataset seal assertion 실패")

split_seal_descriptor = dataset_seal["upstream"]["split_seal"]
if split_seal_descriptor["sha256"] != prediction_seal["split_seal_sha256"]:
    raise AssertionError("prediction과 upstream split seal SHA 불일치")
split_seal_path = verified_download_descriptor(
    split_seal_descriptor, f"split_seal_{SPLIT_VERSION}.json"
)
split_seal = json.loads(split_seal_path.read_text(encoding="utf-8"))
if split_seal.get("output_version") != SPLIT_VERSION:
    raise AssertionError("split seal output version 불일치")
split_assertions = split_seal.get("assertions", {})
if not split_assertions or not all(split_assertions.values()):
    raise AssertionError("split seal assertion 실패")

manifest_seal_path = verified_download_descriptor(
    dataset_seal["upstream"]["manifest_seal"],
    f"manifest_seal_{RAW_MANIFEST_VERSION}.json",
)
manifest_seal = json.loads(manifest_seal_path.read_text(encoding="utf-8"))
if manifest_seal.get("output_version") != RAW_MANIFEST_VERSION:
    raise AssertionError("manifest seal output version 불일치")
if not all(manifest_seal.get("validation", {}).get("hard_assertions", {}).values()):
    raise AssertionError("manifest seal hard assertion 실패")

private_paths = {
    "key_map": verified_download_descriptor(
        dataset_seal["private"]["test_key_map"], "test_key_map.parquet"
    ),
    "truth": verified_download_descriptor(
        split_seal["artifacts"]["test_truth"], f"test_truth_{SPLIT_VERSION}.parquet"
    ),
    "split_seal": split_seal_path,
    "manifest_seal": manifest_seal_path,
    "raw_manifest": verified_download_descriptor(
        manifest_seal["artifacts"]["raw_manifest"],
        f"raw_manifest_{RAW_MANIFEST_VERSION}.parquet",
    ),
}
print("private descriptors pinned by generation/bytes/SHA: PASS")

key_map = pd.read_parquet(private_paths["key_map"])
truth = pd.read_parquet(private_paths["truth"])
assert list(key_map.columns) == ["sample_id", "stem"]
assert list(truth.columns) == ["stem", "waterlevel_m"]
assert len(key_map) == len(truth) == EXPECTED_TEST_N
assert key_map["sample_id"].is_unique and key_map["stem"].is_unique
assert truth["stem"].is_unique

joined = predictions.merge(
    key_map, on="sample_id", how="outer", validate="one_to_one", indicator="_map"
)
assert joined["_map"].eq("both").all()
joined = joined.drop(columns="_map").merge(
    truth, on="stem", how="outer", validate="one_to_one", indicator="_truth"
)
assert joined["_truth"].eq("both").all()
joined = joined.drop(columns="_truth")
assert len(joined) == EXPECTED_TEST_N
assert joined["type"].eq(joined["stem"].str[0]).all()
assert joined["type"].value_counts().to_dict() == EXPECTED_TEST_TYPES
assert np.isfinite(joined[["pred_m", "waterlevel_m"]].to_numpy(float)).all()
joined = joined.merge(
    diagnostics, on="sample_id", how="left", validate="one_to_one"
)
assert joined["fallback_used"].notna().all()
assert joined["route_component"].notna().all()
assert joined["route_component"].eq(joined["type"].map(route_by_type)).all()
print("PRIVATE JOIN HARD GATE: PASS")


## 3. 전체·8유형 RMSE 계산

전체 RMSE는 모든 test 행의 squared error 평균에서 직접 계산합니다. 8개 유형 RMSE의 산술평균을
전체 RMSE로 사용하지 않습니다.


In [ ]:
joined["error_m"] = joined["pred_m"] - joined["waterlevel_m"]
joined["abs_error_m"] = joined["error_m"].abs()

def summarize(part, label):
    error = part["error_m"].to_numpy(float)
    return {
        "scope": label,
        "n": int(len(part)),
        "rmse_m": float(np.sqrt(np.mean(error ** 2))),
        "mae_m": float(np.mean(np.abs(error))),
        "bias_m": float(np.mean(error)),
        "p95_abs_error_m": float(np.quantile(np.abs(error), 0.95)),
    }

metric_rows = [summarize(joined, "ALL")]
metric_rows.extend(
    summarize(joined.loc[joined["type"].eq(type_name)], type_name)
    for type_name in TYPE_ORDER
)
metrics = pd.DataFrame(metric_rows)
overall_rmse = float(metrics.loc[metrics["scope"].eq("ALL"), "rmse_m"].iloc[0])
by_type = metrics.loc[metrics["scope"].isin(TYPE_ORDER)].copy()
weighted_mse = float(
    (by_type["n"] * by_type["rmse_m"] ** 2).sum() / by_type["n"].sum()
)
assert np.isclose(overall_rmse ** 2, weighted_mse, rtol=1e-12, atol=1e-14)
display(metrics)
print("overall RMSE (m):", overall_rmse)


## 4. prediction 봉인 후 사후 metadata 결합

obstruction과 duplicate target conflict는 오류 가설을 만드는 사후 정보입니다. 모델 입력이나
후보 선택에는 사용하지 않았습니다. 자동 taxonomy는 원인을 확정하는 값이 아니라 검토 우선순위입니다.


In [ ]:
raw_manifest_path = private_paths["raw_manifest"]
raw_schema = pq.ParquetFile(raw_manifest_path).schema_arrow.names
desired = [
    "stem", "waterlevel_m", "image_sha256", "obstruction",
    "mokja_count", "valid_mokja_count", "bbox_out_of_range",
]
raw_columns = [name for name in desired if name in raw_schema]
raw = pd.read_parquet(raw_manifest_path, columns=raw_columns)
assert raw["stem"].is_unique

conflict_by_sha = (
    raw.groupby("image_sha256", observed=True)["waterlevel_m"].nunique().gt(1)
    if {"image_sha256", "waterlevel_m"}.issubset(raw.columns)
    else pd.Series(dtype=bool)
)
if not conflict_by_sha.empty:
    raw["duplicate_target_conflict"] = raw["image_sha256"].map(conflict_by_sha).fillna(False)
else:
    raw["duplicate_target_conflict"] = False
metadata_columns = [
    name for name in [
        "stem", "obstruction", "mokja_count", "valid_mokja_count",
        "bbox_out_of_range", "duplicate_target_conflict",
    ] if name in raw.columns
]
joined = joined.merge(
    raw[metadata_columns], on="stem", how="left", validate="one_to_one"
)
assert len(joined) == EXPECTED_TEST_N

def normalized_tags(value):
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (list, tuple)):
        return [str(item) for item in value if str(item)]
    if pd.isna(value):
        return []
    return [str(value)]

type_quantiles = joined.groupby("type", observed=True)["waterlevel_m"].quantile([0.1, 0.9]).unstack()
occlusion_tags = {"수풀", "부착물", "부유물", "사람", "반투명"}
water_tags = {"거센물살", "와류", "탁류", "거울반사", "달빛반사", "야간", "안개"}

def hypothesis_tags(row):
    hypotheses = []
    tags = set(normalized_tags(row.get("obstruction", [])))
    if bool(row.get("fallback_used", False)):
        hypotheses.extend(["detector_miss", "fallback_used"])
    if bool(row.get("duplicate_target_conflict", False)):
        hypotheses.append("label_uncertainty")
    low, high = type_quantiles.loc[row["type"], [0.1, 0.9]]
    if row["type"] in {"F", "G", "H"} or row["waterlevel_m"] <= low or row["waterlevel_m"] >= high:
        hypotheses.append("rare_type_or_extreme_level")
    if tags.intersection(occlusion_tags):
        hypotheses.append("occlusion")
    if tags.intersection(water_tags):
        hypotheses.append("waterline_ambiguity")
    if not hypotheses:
        hypotheses.append("requires_visual_review")
    return ";".join(dict.fromkeys(hypotheses))

joined["hypothesis_tags"] = joined.apply(hypothesis_tags, axis=1)

def summarize_slice(frame, dimension, value):
    error = frame["error_m"].to_numpy(float)
    return {
        "dimension": dimension,
        "value": str(value),
        "n": int(len(frame)),
        "rmse_m": float(np.sqrt(np.mean(error ** 2))),
        "mae_m": float(np.mean(np.abs(error))),
    }

slice_rows = []
for value, part in joined.groupby("route_component", dropna=False):
    slice_rows.append(summarize_slice(part, "route_component", value))
for value, part in joined.groupby("fallback_used", dropna=False):
    slice_rows.append(summarize_slice(part, "fallback_used", value))
joined["candidate_count_slice"] = pd.cut(
    joined["n_candidates"].fillna(0).astype(int),
    bins=[-1, 0, 1, 3, float("inf")],
    labels=["0", "1", "2-3", "4+"],
).astype(str)
for value, part in joined.groupby("candidate_count_slice", dropna=False):
    slice_rows.append(summarize_slice(part, "candidate_count", value))
joined["detector_confidence_slice"] = pd.cut(
    joined["detector_conf"],
    bins=[-float("inf"), 0.10, 0.25, 0.50, float("inf")],
    labels=["<=0.10", "0.10-0.25", "0.25-0.50", ">0.50"],
).astype(str)
joined.loc[joined["detector_conf"].isna(), "detector_confidence_slice"] = "not_applicable"
for value, part in joined.groupby("detector_confidence_slice", dropna=False):
    slice_rows.append(summarize_slice(part, "detector_confidence", value))
joined["level_decile_within_type"] = (
    joined.groupby("type", observed=True)["waterlevel_m"]
    .transform(lambda values: pd.qcut(values.rank(method="first"), 10, labels=False))
    .astype(int)
)
for (type_name, decile), part in joined.groupby(
    ["type", "level_decile_within_type"], observed=True
):
    slice_rows.append(summarize_slice(part, "type_level_decile", f"{type_name}:D{decile}"))
if "obstruction" not in joined.columns:
    joined["obstruction"] = [[] for _ in range(len(joined))]
def real_obstruction_tags(value):
    return [
        tag for tag in normalized_tags(value)
        if tag.strip() and tag.strip() not in {"해당없음", "없음", "none", "None"}
    ]
joined["has_obstruction"] = joined["obstruction"].map(real_obstruction_tags).map(bool)
for value, part in joined.groupby("has_obstruction", dropna=False):
    slice_rows.append(summarize_slice(part, "has_obstruction", value))
obstruction_long = joined[["sample_id", "error_m", "obstruction"]].copy()
obstruction_long["obstruction"] = obstruction_long["obstruction"].map(real_obstruction_tags)
obstruction_long = obstruction_long.explode("obstruction")
obstruction_long = obstruction_long.loc[obstruction_long["obstruction"].notna()]
for value, part in obstruction_long.groupby("obstruction", dropna=False):
    if len(part) >= 10:
        slice_rows.append(summarize_slice(part, "obstruction", value))
performance_slices = pd.DataFrame(slice_rows).sort_values(
    ["dimension", "rmse_m"], ascending=[True, False], kind="mergesort"
)
display(performance_slices)


## 5. 재현 가능한 오류 표본 선택

표본 선택 규칙은 다음과 같습니다.

- 전체 절대오차 상위 20
- 유형별 절대오차 상위 5
- 유형별 상위 10% 오류에서 seed 42로 3개
- fallback 사용 표본의 절대오차 상위 10

같은 표본이 여러 규칙에 걸리면 `selection_reason`을 합칩니다.


In [ ]:
selection_reasons = defaultdict(set)

def add_reason(sample_ids, reason):
    for sample_id in sample_ids:
        selection_reasons[str(sample_id)].add(reason)

add_reason(
    joined.nlargest(20, "abs_error_m")["sample_id"],
    "global_top20",
)
for type_index, type_name in enumerate(TYPE_ORDER):
    part = joined.loc[joined["type"].eq(type_name)]
    add_reason(part.nlargest(5, "abs_error_m")["sample_id"], f"{type_name}_top5")
    threshold = part["abs_error_m"].quantile(0.90)
    high = part.loc[part["abs_error_m"].ge(threshold)]
    sampled = high.sample(
        n=min(3, len(high)), random_state=42 + type_index, replace=False
    )
    add_reason(sampled["sample_id"], f"{type_name}_top10pct_seed42")
fallback_rows = joined.loc[joined["fallback_used"].astype(bool)]
add_reason(
    fallback_rows.nlargest(min(10, len(fallback_rows)), "abs_error_m")["sample_id"],
    "fallback_top_error",
)

error_selection = joined.loc[
    joined["sample_id"].isin(selection_reasons)
].copy()
error_selection["selection_reason"] = error_selection["sample_id"].map(
    lambda value: ";".join(sorted(selection_reasons[value]))
)
error_selection = error_selection.sort_values(
    ["abs_error_m", "sample_id"], ascending=[False, True], kind="mergesort"
)
display(error_selection[[
    "stem", "type", "waterlevel_m", "pred_m", "error_m",
    "route_component", "fallback_used", "hypothesis_tags", "selection_reason",
]].head(30))
print("selected error cases:", len(error_selection))


## 6. 그림 생성

montage는 가장 큰 오류 20개를 익명 shard byte에서 찾습니다. `hypothesis_tags`는 시각 검토 후
확인·수정할 가설입니다.


In [ ]:
rmse_figure_path = LOCAL_FIGURES / "test_rmse_by_type.png"
fig, ax = plt.subplots(figsize=(10, 5))
plot_rows = metrics.loc[metrics["scope"].isin(TYPE_ORDER)]
ax.bar(plot_rows["scope"], plot_rows["rmse_m"], color="#4C78A8")
ax.axhline(overall_rmse, color="#E45756", linestyle="--", label=f"ALL {overall_rmse:.3f}m")
ax.set_xlabel("Mokja type")
ax.set_ylabel("RMSE (meter)")
ax.set_title("Mission 3 test RMSE by mokja type")
ax.legend()
fig.tight_layout()
fig.savefig(rmse_figure_path, dpi=160)
plt.show()

residual_figure_path = LOCAL_FIGURES / "test_residual_distribution.png"
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(joined["error_m"], bins=80, color="#72B7B2", alpha=0.9)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Prediction - truth (meter)")
ax.set_ylabel("Images")
ax.set_title("Mission 3 test residual distribution")
fig.tight_layout()
fig.savefig(residual_figure_path, dpi=160)
plt.show()


In [ ]:
montage_path = LOCAL_FIGURES / "top20_error_montage.png"
if BUILD_ERROR_MONTAGE:
    montage_rows = error_selection.head(20).copy()
    wanted = set(montage_rows["sample_id"])
    image_bytes_by_id = {}
    for shard_number, descriptor in enumerate(inference_image_seal["test_shards"]):
        if not wanted - set(image_bytes_by_id):
            break
        _, object_name = parse_gs_uri(descriptor["uri"])
        local_shard = LOCAL_INPUT / f"montage-{shard_number:04d}.tar"
        download_object(
            BUCKET, object_name, local_shard, generation=descriptor["generation"]
        )
        if sha256_file(local_shard) != descriptor["sha256"]:
            raise AssertionError("montage shard SHA 불일치")
        with tarfile.open(local_shard, "r:") as handle:
            member_names = {member.name for member in handle.getmembers()}
            for sample_id in wanted - set(image_bytes_by_id):
                name = f"{sample_id}.jpg"
                if name in member_names:
                    image_bytes_by_id[sample_id] = handle.extractfile(name).read()
        local_shard.unlink(missing_ok=True)
    if set(image_bytes_by_id) != wanted:
        raise AssertionError("montage image 완전성 실패")

    fig, axes = plt.subplots(4, 5, figsize=(18, 12))
    for ax, row in zip(axes.ravel(), montage_rows.itertuples()):
        image = Image.open(io.BytesIO(image_bytes_by_id[row.sample_id])).convert("RGB")
        ax.imshow(image)
        for box in json.loads(getattr(row, "candidate_boxes_xyxy_json", "[]")):
            x1, y1, x2, y2 = map(float, box)
            ax.add_patch(Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                fill=False, edgecolor="#ff3b30", linewidth=1.3,
            ))
        ax.set_title(
            f"{row.type} truth={row.waterlevel_m:.1f} pred={row.pred_m:.1f}\n"
            f"error={row.error_m:+.2f}m | {row.hypothesis_tags}",
            fontsize=8,
        )
        ax.axis("off")
    for ax in axes.ravel()[len(montage_rows):]:
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(montage_path, dpi=150)
    plt.show()


## 7. private 결과 게시와 evaluation seal

residual에는 정답이 포함되므로 모든 산출물을 `eval_private` 아래에 저장합니다.


In [ ]:
metrics_path = LOCAL_OUTPUT / "test_metrics.csv"
residuals_path = LOCAL_OUTPUT / "test_residuals.parquet"
error_selection_path = LOCAL_OUTPUT / "error_selection.parquet"
performance_slices_path = LOCAL_OUTPUT / "performance_slices.csv"
metrics.to_csv(metrics_path, index=False, encoding="utf-8-sig")
joined.to_parquet(residuals_path, index=False, compression="zstd")
error_selection.to_parquet(error_selection_path, index=False, compression="zstd")
performance_slices.to_csv(performance_slices_path, index=False, encoding="utf-8-sig")

artifact_specs = {
    "metrics": (metrics_path, "test_metrics.csv", "m3_private_test_metrics"),
    "residuals": (residuals_path, "test_residuals.parquet", "m3_private_test_residuals"),
    "error_selection": (
        error_selection_path, "error_selection.parquet", "m3_private_error_selection"
    ),
    "performance_slices": (
        performance_slices_path, "performance_slices.csv", "m3_private_error_slices"
    ),
    "rmse_figure": (
        rmse_figure_path, "figures/test_rmse_by_type.png", "m3_private_figure"
    ),
    "residual_figure": (
        residual_figure_path, "figures/test_residual_distribution.png", "m3_private_figure"
    ),
}
if BUILD_ERROR_MONTAGE:
    artifact_specs["error_montage"] = (
        montage_path, "figures/top20_error_montage.png", "m3_private_figure"
    )

descriptors = {}
for name, (local_path, relative_name, artifact_type) in artifact_specs.items():
    descriptors[name] = upload_create_only(
        BUCKET, local_path, f"{EVAL_PREFIX}/{relative_name}", artifact_type
    )

metric_payload = {
    row["scope"]: {
        "n": int(row["n"]),
        "rmse_m": float(row["rmse_m"]),
        "mae_m": float(row["mae_m"]),
        "bias_m": float(row["bias_m"]),
        "p95_abs_error_m": float(row["p95_abs_error_m"]),
    }
    for row in metrics.to_dict(orient="records")
}
evaluation_seal = {
    "status": "PRIVATE_EVALUATION_COMPLETE",
    "schema_version": "dcc-m3-private-evaluation-seal-v2",
    "evaluation_id": EVALUATION_ID,
    "prediction_run_id": PREDICTION_RUN_ID,
    "dataset_version": DATASET_VERSION,
    "winner_model_id": prediction_seal["winner_model_id"],
    "winner_kind": prediction_seal["winner_kind"],
    "routing_sha256": prediction_seal["routing_sha256"],
    "route_counts": prediction_seal["route_counts"],
    "training_scope": prediction_seal["training_scope"],
    "prediction_seal_sha256": prediction_seal_sha256,
    "prediction_sha256": prediction_seal["prediction"]["sha256"],
    "truth_sha256": sha256_file(private_paths["truth"]),
    "key_map_sha256": sha256_file(private_paths["key_map"]),
    "row_count": EXPECTED_TEST_N,
    "metrics": metric_payload,
    "artifacts": descriptors,
    "assertions": {
        "prediction_verified_before_truth": True,
        "join_one_to_one_complete": len(joined) == EXPECTED_TEST_N,
        "type_counts_exact": joined["type"].value_counts().to_dict() == EXPECTED_TEST_TYPES,
        "route_counts_exact": (
            joined["route_component"].value_counts().to_dict()
            == expected_route_counts
        ),
        "routing_contract_verified_before_truth": True,
        "overall_weighted_identity": bool(
            np.isclose(overall_rmse ** 2, weighted_mse, rtol=1e-12, atol=1e-14)
        ),
        "error_selection_rule_fixed": True,
        "prediction_not_modified": sha256_file(prediction_path) == prediction_seal["prediction"]["sha256"],
    },
}
evaluation_descriptor = upload_json_create_only(
    BUCKET,
    evaluation_seal,
    f"{EVAL_PREFIX}/evaluation_seal.json",
    "m3_private_evaluation_seal",
)
print("PRIVATE EVALUATION COMPLETE")
print("winner:", prediction_seal["winner_model_id"])
print("overall RMSE (m):", overall_rmse)
print("evaluation seal:", evaluation_descriptor["uri"])


## Mission 3 답안에 옮길 값

1. 03D leaderboard의 3개 이상 후보와 validation RMSE
2. selection seal의 승자와 선택 이유
3. 이 노트북의 `ALL + A/B/C/E/F/G/H/S` test RMSE
4. error montage와 `error_selection.parquet`의 구체 사례
5. `hypothesis_tags`는 관찰 근거와 함께 검토하고 확정 원인처럼 표현하지 않음

이 노트북은 full deployment prediction만 평가합니다. `training_scope=full`을 최종 seal에서
확인하고 수치를 답안에 옮기십시오.
